# ARK-020 V4 — A1.1 hardened operator launcher

This notebook is pinned to the CPU-CI-green A1.1 executable. Run cells in order. Cell 0 mounts Drive, fetches and detach-checkouts the exact executable commit, and runs the fail-closed hardened scan. Cell 1 runs the A1/A1.1 durability regressions plus the inherited V4 suite on the live Colab runtime. Cell 2 starts/resumes the campaign only if both gates passed. Cell 3 prints receipts/bundle status.


In [ ]:
import json, subprocess, sys, os, shutil
from pathlib import Path

PINNED_EXECUTABLE_COMMIT = '5bc4a87bf5f7f7c47b98639eb29e18e56ad41b20'
EXECUTABLE_BRANCH = 'codex/arkenstone-v4-durability-hardening'
REPO = '/content/An-Ra-the-new-AGI-ark020v4-a11'
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
SCAN_GATE_PASS = False
SAFE_ACTION = None

print('=== ARK-020 V4 A1.1 CELL 0: MOUNT / PIN / HARDENED SCAN ===')
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = Path('/content/drive/MyDrive').is_dir()
except Exception as exc:
    DRIVE_OK = False
    print('DRIVE MOUNT FAILED:', repr(exc))
if not DRIVE_OK:
    raise RuntimeError('SAFE ACTION: STOP — DRIVE UNAVAILABLE')

if os.path.exists(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    shutil.rmtree(REPO)
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','100','--branch',EXECUTABLE_BRANCH,REPO_URL,REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'remote','set-url','origin',REPO_URL], check=True)
    subprocess.run(['git','-C',REPO,'fetch','--depth','100','origin',EXECUTABLE_BRANCH], check=True)
subprocess.run(['git','-C',REPO,'reset','--hard'], check=True)
subprocess.run(['git','-C',REPO,'clean','-fd'], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_EXECUTABLE_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_EXECUTABLE_COMMIT, (head, PINNED_EXECUTABLE_COMMIT)
print('PINNED A1.1 EXECUTABLE OK:', head)

runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4_hardened.py')
print('HARDENED RUNNER:', runner)
scan = subprocess.run([sys.executable, runner, '--mode','scan','--drive-ok','True'], cwd=REPO, capture_output=True, text=True)
print(scan.stdout)
if scan.stderr.strip():
    print('--- scan stderr ---')
    print(scan.stderr)
if scan.returncode != 0:
    raise RuntimeError(f'SAFE ACTION: STOP — SCAN COMMAND FAILED (return code {scan.returncode})')
marker = '@@SCAN_JSON@@'
marker_line = next((ln for ln in scan.stdout.splitlines() if ln.startswith(marker)), None)
if marker_line is None:
    raise RuntimeError('SAFE ACTION: STOP — SCAN DID NOT EMIT @@SCAN_JSON@@')
scan_info = json.loads(marker_line[len(marker):])
SAFE_ACTION = scan_info.get('SAFE_ACTION')
print('PARSED SAFE ACTION:', SAFE_ACTION)
print('HARDENED GATE:', scan_info.get('HARDENED_GATE'))
if scan_info.get('HARDENED_ERRORS'):
    print('HARDENED ERRORS:', json.dumps(scan_info['HARDENED_ERRORS'], indent=2))
if SAFE_ACTION not in {'START NEW CAMPAIGN','RESUME'}:
    raise RuntimeError('Not safe to continue: ' + str(SAFE_ACTION))
SCAN_GATE_PASS = True
print('CELL 0 GATE: PASS')


In [ ]:
import subprocess, sys, torch, os

assert globals().get('SCAN_GATE_PASS') is True, 'Run Cell 0 successfully first.'
TEST_GATE_PASS = False
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('Python:', sys.version)
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

paths = [
    'experiments/ARK-020-V4/ark020_v4_core.py',
    'experiments/ARK-020-V4/run_ark020_v4.py',
    'experiments/ARK-020-V4/ark020_v4_durability.py',
    'experiments/ARK-020-V4/ark020_v4_a1_guardrails.py',
    'experiments/ARK-020-V4/run_ark020_v4_hardened.py',
    'tests/test_ark020_v4.py',
    'tests/test_ark020_v4_durability.py',
    'tests/test_ark020_v4_a1_guardrails.py',
]
for p in paths:
    subprocess.run([sys.executable, '-m', 'py_compile', os.path.join(REPO, p)], check=True)
print('compile gate: PASS on', len(paths), 'files')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
print('Running A1/A1.1 durability regressions with live output...')
cmd1 = [sys.executable, '-m', 'pytest', 'tests/test_ark020_v4_durability.py', 'tests/test_ark020_v4_a1_guardrails.py', '-q', '-ra']
r1 = subprocess.run(cmd1, cwd=REPO)
if r1.returncode != 0:
    raise RuntimeError(f'A1/A1.1 durability tests failed (return code {r1.returncode}) — DO NOT RUN campaign')

print('Running inherited V4 suite with live output...')
cmd2 = [sys.executable, '-m', 'pytest', 'tests/test_ark020_v4.py', '-q', '-ra']
r2 = subprocess.run(cmd2, cwd=REPO)
if r2.returncode != 0:
    raise RuntimeError(f'inherited V4 tests failed (return code {r2.returncode}) — DO NOT RUN campaign')
TEST_GATE_PASS = True
print('CELL 1 GATE: PASS — hardened + inherited V4 suites passed on this Colab runtime')


In [ ]:
# Full campaign. Exact-resumable: on a new T4 session rerun Cell 0 -> Cell 1 -> Cell 2.
import os, subprocess, sys, json
from pathlib import Path

assert globals().get('SCAN_GATE_PASS') is True, 'Cell 0 gate not passed.'
assert globals().get('TEST_GATE_PASS') is True, 'Cell 1 test gate not passed.'
runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4_hardened.py')
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
env = dict(os.environ); env['PYTHONUNBUFFERED'] = '1'
print('=== STARTING / RESUMING ARK-020 V4 A1.1 ===', flush=True)
proc = subprocess.run([sys.executable, runner, '--mode','all'], cwd=REPO, env=env)
print('CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    failure = root / 'ARK-020_V4_FAILURE.json'
    if failure.exists():
        print('\n===== REAL V4 FAILURE RECEIPT =====')
        try:
            f = json.loads(failure.read_text())
            print('EXCEPTION:', f.get('exception'))
            print('MESSAGE:', f.get('message'))
            print(f.get('traceback', failure.read_text()))
        except Exception:
            print(failure.read_text())
    raise RuntimeError('ARK-020 V4 A1.1 child process failed — traceback printed above')
result = root / 'ARK-020_V4_RESULT.json'
session = root / 'SESSION_STATE.json'
if result.exists():
    r = json.loads(result.read_text())
    print('SCIENTIFIC CAMPAIGN STATUS:', r.get('status'))
    print('VERDICT:', r.get('decision', {}).get('verdict', r.get('verdict')))
elif session.exists():
    s = json.loads(session.read_text())
    print('SESSION STATUS:', s.get('status'))
    print('MESSAGE:', s.get('message'))
    if s.get('status') == 'PARTIAL_SESSION':
        print('Expected multi-session stop. On next T4: rerun Cell 0 -> Cell 1 -> Cell 2.')
else:
    print('Process returned 0 but no result/session receipt found. Inspect Drive before rerunning.')


In [ ]:
from pathlib import Path
import json
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
print('CAMPAIGN ROOT:', root)
for name in ['EXECUTABLE_IDENTITY_A1.json','PREEXECUTION_GATE.json','EXACT_RESUME_SMOKE_V4.json','EXACT_RESUME_SMOKE_V4_A1.json','SESSION_STATE.json','ARK-020_V4_RESULT.json','ARK-020_V4_FAILURE.json']:
    p = root / name
    if p.exists():
        print('\n===', name, '===')
        print(p.read_text()[:8000])
final_zip = root / 'ARKENSTONE_ARK020_V4_CONTINUAL_RESULTS.zip'
partial_zip = root / 'ARKENSTONE_ARK020_V4_CONTINUAL_PARTIAL.zip'
z = final_zip if final_zip.exists() else partial_zip if partial_zip.exists() else None
print('\nBUNDLE:', z if z else 'none yet')
